# Chapter 2. Molecular Mechanics

Part 2: force fields, conformational energy, and geometry optimization

## 2.2. Molecular Mechanics

Molecular mechanics (MM) evaluates an approximate potential energy from a molecular structure and a parameterized model. It is useful for preparing geometries and exploring conformations.

**Learning objectives**

- Explain what a force field specifies and where its predictions apply.
- Build a reproducible ethane torsional energy scan.
- Distinguish energy gradients from forces, and local minima from global minima.
- Minimize molecules with UFF/MMFF94, check convergence, and inspect a minimization trajectory.

**Run this notebook:** use the environment in the repository README, then **Restart Kernel and Run All Cells**. No other notebook needs to run first. Core examples use inline structures and run offline; the optional file audit and 3D viewer are labeled at the end.

Our examples describe isolated molecules with fixed connectivity, formal charge, and stereochemistry. These UFF/MMFF calculations do not describe bond making/breaking, electronic excitations, or solvent automatically. Chemical reactions require an appropriate reactive or quantum model; Chapter 3 extends these force models to molecular dynamics; Chapter 4 introduces quantum mechanics.

### What does “lower energy” mean here?

Imagine a landscape whose location is the molecule's geometry. A **single point** asks the height at one location; a **scan** follows a chosen route; a **minimization** moves downhill. A **local minimum** is a valley lower than nearby points; the **global minimum** is the lowest valley over all allowed coordinates. This analogy describes the selected mathematical model, not experimental truth.

| Earlier skill | New question |
| --- | --- |
| SMILES tells us the graph | Does the model have parameters for this chemistry? |
| A conformer supplies coordinates | What energy does this model assign to this shape? |
| A torsion setter rotates a group | How does the energy change along that controlled edit? |
| An optimizer changes many coordinates | Did it converge, and do other starts reach different valleys? |

**Core route:** ethane rotation and its energy curve → the spring/force picture → the three-start butane application → one checked optimizer example. The catalog of detailed potential terms, glucose trajectory internals, and file export are **deeper reading**. The code remains executable in order; learning the interpretation is more useful initially than memorizing every helper call.

**Predict:** if the force points toward increasing bond length, is the derivative of energy positive or negative at that point?

### 2.2.1. Energy Functions

For fixed chemical identity and a chosen force field, an energy function maps the $3N$ Cartesian coordinates to a scalar:

$$E(\mathbf q;\mathbf p),\qquad
\mathbf q=(x_1,y_1,z_1,\ldots,x_N,y_N,z_N).$$

Here $\mathbf p$ denotes the force-field parameters. A **single-point calculation** evaluates $E$ at the supplied coordinates without moving atoms. A **geometry optimization** searches for coordinates with lower energy.

RDKit's UFF/MMFF energies are in **kcal/mol** and coordinates in **angstroms** ($1\ \mathrm{kcal}=4.184\ \mathrm{kJ}$). Compare relative energies only within a consistent model and chemical system. A raw MM energy is not an electronic total energy, an experimental heat of formation, or a free energy. Temperature and entropy are absent from these minimizations. See the [RDKit force-field API](https://www.rdkit.org/docs/source/rdkit.ForceField.rdForceField.html).

### 2.2.2. Molecular Mechanics

An all-atom MM model represents each atom by coordinates and interaction parameters; hydrogen atoms are usually explicit. The molecular graph defines bonded interactions, while selected atom pairs interact through space. Parameters encode chemical environment (atom types), equilibrium geometries, and interaction strengths. Having a valid SMILES string does not guarantee that a force field has parameters for every atom in it.

### 2.2.3. Force Field

A **force field** includes functional forms, parameter values, atom-typing rules, and rules for excluding or scaling selected interactions.

| Family | Context and limitation |
|---|---|
| UFF (Universal Force Field) | Broad element coverage based on general parameter rules. Coverage does not guarantee quantitative accuracy for a particular bonding environment. |
| MMFF94 / MMFF94s (Merck Molecular Force Field) | Often used for small organic molecules. Specify the variant and check parameter availability. We use **MMFF94** consistently here. |
| AMBER, CHARMM, GROMOS, OPLS | Families used in biomolecular or condensed-phase modeling. A specific version, compatible small-molecule parameters, and solvent model must be selected. |
| DREIDING, COMPASS | Other families encountered in molecular and materials modeling; their functional forms and parameter scope differ. |

UFF and MMFF are available through RDKit in this notebook. The other names are context, not interchangeable options in the RDKit calls below. Select a model for its validation against the chemistry and property of interest, not just its age or name.

Primary descriptions: [Rappe et al., UFF (1992)](https://doi.org/10.1021/ja00051a040), [Halgren, MMFF94 (1996)](https://onlinelibrary.wiley.com/doi/abs/10.1002/%28SICI%291096-987X%28199604%2917%3A5/6%3C490%3A%3AAID-JCC1%3E3.0.CO%3B2-P). RDKit documents [parameter checks and minimization results](https://www.rdkit.org/docs/source/rdkit.Chem.rdForceFieldHelpers.html).

#### Worked example: the torsional energy of ethane

First generate 3D coordinates with **ETKDGv3** (distance geometry with empirical torsion preferences), then minimize with UFF. Embedding proposes a geometry; it does not guarantee a force-field minimum. We set the random seed and check the return value. The same software version and seed make this small example reproducible; numerical results may change slightly across RDKit releases. See [RDKit embedding parameters](https://www.rdkit.org/docs/source/rdkit.Chem.rdDistGeom.html).

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from rdkit import Chem, rdBase
from rdkit.Chem import rdDistGeom, rdMolTransforms, rdMolDescriptors
from rdkit.Chem import rdForceFieldHelpers as FF
from rdkit.Chem import rdtrajectory

SEED = 42
print(f"RDKit {rdBase.rdkitVersion}; NumPy {np.__version__}")


def embed_smiles(smiles, seed=SEED):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"Invalid SMILES: {smiles}")
    mol = Chem.AddHs(mol)
    params = rdDistGeom.ETKDGv3()
    params.randomSeed = seed
    params.numThreads = 1
    if rdDistGeom.EmbedMolecule(mol, params) != 0:
        raise RuntimeError("3D embedding failed; inspect the structure and parameters.")
    return mol


def make_force_field(mol, model="UFF"):
    if mol.GetNumConformers() == 0:
        raise ValueError("A force field needs coordinates.")
    if model == "UFF":
        if not FF.UFFHasAllMoleculeParams(mol):
            raise ValueError("UFF parameters are missing for this molecule.")
        ff = FF.UFFGetMoleculeForceField(mol)
    elif model == "MMFF94":
        if not FF.MMFFHasAllMoleculeParams(mol):
            raise ValueError("MMFF parameters are missing for this molecule.")
        properties = FF.MMFFGetMoleculeProperties(mol, mmffVariant="MMFF94")
        if properties is None:
            raise ValueError("MMFF94 atom typing failed.")
        ff = FF.MMFFGetMoleculeForceField(mol, properties)
    else:
        raise ValueError("Choose 'UFF' or 'MMFF94'.")
    if ff is None:
        raise RuntimeError("Force-field construction failed.")
    ff.Initialize()
    return ff


def plot_molecule(mol, ax, title):
    """Display actual 3D coordinates using only Matplotlib."""
    xyz = mol.GetConformer().GetPositions()
    colors = {"C": "#334155", "H": "#93c5fd", "O": "#dc2626"}
    for bond in mol.GetBonds():
        pair = xyz[[bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()]]
        ax.plot(*pair.T, color="0.5", linewidth=1.5)
    for atom, point in zip(mol.GetAtoms(), xyz):
        ax.scatter(*point, color=colors.get(atom.GetSymbol(), "#a855f7"), s=35)
    center = xyz.mean(axis=0)
    radius = max(np.ptp(xyz, axis=0).max() / 2, 0.5) + 0.3
    ax.set(xlim=(center[0]-radius, center[0]+radius),
           ylim=(center[1]-radius, center[1]+radius),
           zlim=(center[2]-radius, center[2]+radius), title=title)
    ax.set_box_aspect((1, 1, 1))
    ax.set_axis_off()

In [ ]:
ethane = embed_smiles("CC")
ethane_ff = make_force_field(ethane, "UFF")
ethane_embed_energy = ethane_ff.CalcEnergy()
status = ethane_ff.Minimize(maxIts=1000)
if status != 0:
    raise RuntimeError("The ethane UFF minimization did not converge.")
ethane_min_energy = ethane_ff.CalcEnergy()
print(f"UFF: embedded {ethane_embed_energy:.4f} -> minimized {ethane_min_energy:.4f} kcal/mol")

# Find the terminal H atoms from connectivity rather than assuming their indices.
carbon0, carbon1 = [atom.GetIdx() for atom in ethane.GetAtoms() if atom.GetAtomicNum() == 6]
hydrogen0 = next(a.GetIdx() for a in ethane.GetAtomWithIdx(carbon0).GetNeighbors() if a.GetAtomicNum() == 1)
hydrogen1 = next(a.GetIdx() for a in ethane.GetAtomWithIdx(carbon1).GetNeighbors() if a.GetAtomicNum() == 1)
ethane_torsion = (hydrogen0, carbon0, carbon1, hydrogen1)
print("Selected H-C-C-H atom indices:", ethane_torsion)

In [ ]:
# Construct a known staggered conformer from the minimized reference geometry.
staggered = Chem.Mol(ethane)
rdMolTransforms.SetDihedralDeg(staggered.GetConformer(), *ethane_torsion, 60.0)
staggered_energy = make_force_field(staggered).CalcEnergy()
print(f"Staggered (60 degrees): {staggered_energy:.4f} kcal/mol")

Make an eclipsed conformer from the **same reference** by rotating the methyl group once. This keeps its internal geometry fixed; sequentially setting three H--C--C--H torsions is unnecessary. Calculate a new force field after the coordinate edit.

In [ ]:
eclipsed = Chem.Mol(ethane)
rdMolTransforms.SetDihedralDeg(eclipsed.GetConformer(), *ethane_torsion, 0.0)
fig = plt.figure(figsize=(10, 4))
plot_molecule(staggered, fig.add_subplot(121, projection="3d"), "Staggered: selected torsion 60 degrees")
plot_molecule(eclipsed, fig.add_subplot(122, projection="3d"), "Eclipsed: selected torsion 0 degrees")
plt.tight_layout()
plt.show()

In [ ]:
eclipsed_energy = make_force_field(eclipsed).CalcEnergy()
print(f"Eclipsed (0 degrees): {eclipsed_energy:.4f} kcal/mol")
print(f"Eclipsed - staggered: {eclipsed_energy-staggered_energy:.4f} kcal/mol")
assert eclipsed_energy > staggered_energy

The UFF model places this eclipsed geometry above the staggered geometry. Now sample the full rotation in a **rigid torsion scan**: change the chosen dihedral and leave other internal coordinates unrelaxed. A **relaxed scan** would instead constrain the dihedral and minimize the other coordinates at each point.

The curve below is a total UFF energy along a chosen path, not an isolated torsion term or an experimentally measured free-energy barrier. Unconstrained minimization at every point would usually return to a nearby minimum and erase the intended scan.

In [ ]:
def torsion_energy(reference_mol, torsion, angle_deg, model="UFF"):
    """Evaluate one rigid-scan geometry; never mutate the caller's molecule."""
    trial = Chem.Mol(reference_mol)
    rdMolTransforms.SetDihedralDeg(trial.GetConformer(), *torsion, float(angle_deg))
    return make_force_field(trial, model).CalcEnergy()

In [ ]:
reference_positions = ethane.GetConformer().GetPositions().copy()
angles_deg = np.linspace(0.0, 360.0, 121)
energies = np.array([torsion_energy(ethane, ethane_torsion, angle) for angle in angles_deg])
relative_energies = energies - energies.min()
np.testing.assert_allclose(ethane.GetConformer().GetPositions(), reference_positions)
np.testing.assert_allclose(energies[0], energies[-1], atol=1e-8)
assert np.isfinite(energies).all()

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(angles_deg, relative_energies, color="#2563eb")
ax.set(xlabel="Selected H-C-C-H torsion (degrees)",
       ylabel="Relative UFF energy (kcal/mol)",
       title="Ethane: rigid torsion scan", xticks=np.arange(0, 361, 60))
ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()
print(f"Sampled rigid-scan energy range: {np.ptp(energies):.4f} kcal/mol")

### 2.2.4. Potential Energy

**Core idea:** geometry changes alter energy because different interactions favor different distances and angles. Read the spring figure first. The detailed list of potential terms below is a **deeper reference**, not a list to memorize before using a force field.

The following expressions explain common building blocks. They are **illustrative forms**, not a complete implementation of UFF or MMFF94. Force fields differ in their functions, prefactors, parameter units, and interactions included. Never combine equations and constants from different parameterizations without checking their definitions.

A useful schematic decomposition is

$$E=E_{\rm bonded}+E_{\rm nonbonded}+E_{\rm optional}.$$

Optional terms may include a specified restraint or an implicit-solvent model. They are not included merely because we call a force-field energy function.

Typical bonded terms are

$$E_{\rm bonded}=E_{\rm stretch}+E_{\rm bend}+E_{\rm proper\ torsion}
+E_{\rm improper}+E_{\rm coupling}.$$

Only terms present in the selected force field belong in the sum. **Out-of-plane bending** and **inversion** can be represented by related improper-coordinate terms; they are not automatically two separate contributions to add.

A common nonbonded decomposition is

$$E_{\rm nonbonded}=E_{\rm vdW}+E_{\rm electrostatic}.$$

The eligible pair list matters: directly bonded (1--2) and angle-connected (1--3) pairs are often excluded; torsion-connected (1--4) pairs may be included with special scaling. Use the rules of the chosen force field. Sum each pair **once**, using $i<j$, and never include the self-pair $i=j$.

#### 2.2.4.1. Bond Stretching Energy (E<sub>bonds</sub>)

Near a reference length, a harmonic bond behaves like a spring:

$$E_{\rm stretch}=\sum_{b\in\mathrm{bonds}}\frac12 k_b(r_b-r_{0b})^2.$$

Here $r_b$ is the current length, $r_{0b}$ the reference length, and $k_b$ the stiffness (energy per squared length). Some parameter conventions absorb $1/2$ into $k_b$. A harmonic spring grows without limit when stretched, so it does not model dissociation. See the [GROMACS bonded-interaction definitions](https://manual.gromacs.org/current/reference-manual/functions/bonded-interactions.html).

#### Read energy and force together

The derivative $dE/dr$ is the slope of the energy curve; the radial force $F_r=-dE/dr$ has the opposite sign. In the simple spring below, a compressed bond is pushed toward larger $r$, and a stretched bond toward smaller $r$. The atoms experience opposite vector forces; the plot shows the force associated with increasing their separation.

We choose $r_0=1.50$ angstrom and $k=300$ kcal mol$^{-1}$ angstrom$^{-2}$ **only to illustrate the equation**. These are not claimed UFF/MMFF parameters or a measured bond stiffness. **Predict:** where are energy and force zero? Where is the force largest in magnitude?

In [ ]:
spring_r0 = 1.50  # angstrom; illustrative parameter
spring_k = 300.0  # kcal/(mol angstrom**2); illustrative parameter
spring_r = np.linspace(1.30, 1.70, 201)
spring_energy = 0.5 * spring_k * (spring_r - spring_r0)**2
spring_force = -spring_k * (spring_r - spring_r0)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), constrained_layout=True)
axes[0].plot(spring_r, spring_energy, color="#2563eb")
axes[1].plot(spring_r, spring_force, color="#b45309")
axes[0].set(ylabel="Spring energy (kcal/mol)", title="Energy: cost of displacement")
axes[1].set(ylabel="Radial force (kcal/(mol angstrom))", title="Force: direction of restoration")
for ax in axes:
    ax.axvline(spring_r0, color="0.4", linestyle="--", label="reference length")
    ax.axhline(0, color="0.6", linewidth=0.8)
    ax.set(xlabel="Separation r (angstrom)")
axes[1].annotate("push apart", xy=(1.36, 42), xytext=(1.39, 20),
                 arrowprops={"arrowstyle": "->"})
axes[1].annotate("pull together", xy=(1.64, -42), xytext=(1.48, -20),
                 arrowprops={"arrowstyle": "->"})
plt.show()
assert np.isclose(spring_energy[100], 0) and np.isclose(spring_force[100], 0)

**Explain:** the force is zero at the energy minimum, even though the spring is stiff. Stiffness describes how rapidly the restoring force changes when displaced, not a permanent nonzero force at equilibrium. In a molecule, several terms act at once, so one term's reference length need not equal the final optimized bond length.

#### 2.2.4.2. Angle Bending Energy (E<sub>angles</sub>)

A simple harmonic angle term is

$$E_{\rm bend}=\sum_{a\in\mathrm{angles}}\frac12 k_a(\theta_a-\theta_{0a})^2.$$

Angles in this equation are in **radians**, with $k_a$ in energy per squared radian. An equilibrium parameter for an isolated term is not necessarily the angle in an optimized molecule, where several terms compete. Some force fields use other angle functions.

#### 2.2.4.3. Torsional Energy (E<sub>torsion</sub>)

A periodic torsion can be expressed as

$$E_{\rm torsion}=\sum_{t\in\mathrm{torsions}}\sum_n
V_{tn}\left[1+\cos(n\phi_t-\delta_{tn})\right].$$

$n$ is the periodicity, $\delta_{tn}$ a phase, and $V_{tn}$ the coefficient **under this convention**. Some conventions use $V_{tn}/2$. For a single positive-coefficient term, the maximum-minus-minimum is $2V_{tn}$. The total molecular barrier also depends on other energy terms. The [OpenMM theory guide](https://docs.openmm.org/latest/userguide/theory/02_standard_forces.html#periodictorsionforce) specifies one such convention.

#### 2.2.4.4. Out-of-Plane Bending Energy (E<sub>out of plane</sub>)


An improper coordinate can penalize departure from planarity or help preserve a stereocenter. One possible angular form is

$$E_{\rm improper}=\sum_u\frac12 k_u(\chi_u-\chi_{0u})^2.$$

$\chi_u$ is a defined improper angle, and its atom ordering, reference value, and angular wrapping must match the parameterization. A planar reference often has $\chi_{0u}=0$ under its chosen convention. A displacement-based coordinate would require a **different** force constant and units. See [improper dihedrals in GROMACS](https://manual.gromacs.org/current/reference-manual/functions/bonded-interactions.html#improper-dihedrals).

#### 2.2.4.5. Inversion (E<sub>inversion</sub>)

Inversion describes passage between pyramidal configurations, for example at an amine nitrogen. The stable configurations need not be planar: a planar geometry can lie at the top of an inversion barrier. Consequently, a universal expression $\tfrac12 k\chi^2$ centered at planarity would be misleading for pyramidal inversion. Use the inversion/improper functional form and parameters of the selected model, and verify that this behavior is within its intended scope.

#### 2.2.4.6. Cross Term (E<sub>cross</sub>)

A coupling term describes how one internal coordinate changes the energetic preference of another. For example, an illustrative stretch--bend coupling is

$$E_{\rm stretch\text{-}bend}=\sum_c k_c(r_c-r_{0c})(\theta_c-\theta_{0c}).$$

The sum runs over **defined coupled internal coordinates**, not arbitrary atom pairs. Its units are energy/(length $\times$ radian). MMFF94 includes stretch--bend interactions; see [the original MMFF functional-form documentation](https://hpc.nih.gov/apps/charmm/c39b2html/mmff.html).

#### 2.2.4.7. Van der Waals Energy (E<sub>vdw</sub>)

The Lennard--Jones 12--6 form is

$$E_{\rm vdW}=\sum_{(i,j)\in\mathcal P,\ i<j}s^{\rm vdW}_{ij}\,
4\epsilon_{ij}\left[\left(\frac{\sigma_{ij}}{r_{ij}}\right)^{12}
-\left(\frac{\sigma_{ij}}{r_{ij}}\right)^6\right].$$

$\mathcal P$ contains eligible pairs and $s^{\rm vdW}_{ij}$ is a pair scaling factor. $\epsilon_{ij}$ is a well depth; $\sigma_{ij}$ is the zero-crossing distance. The minimum is at $r=2^{1/6}\sigma$ with energy $-\epsilon$. The steep repulsive part discourages overlap, and the longer-range attractive part approximates dispersion. Pair parameters require specified combining rules. MMFF94 uses a buffered 14--7 form instead of this equation. See [OpenMM's Lennard--Jones definition](https://docs.openmm.org/latest/userguide/theory/02_standard_forces.html#lennard-jones-interaction) and [MMFF functional forms](https://hpc.nih.gov/apps/charmm/c39b2html/mmff.html).

#### 2.2.4.8. Electrostatic Energy (E<sub>electrostatic</sub>)

An unscreened point-charge model with a uniform relative permittivity has

$$E_{\rm electrostatic}=\sum_{(i,j)\in\mathcal P,\ i<j}s^{\rm elec}_{ij}
\frac{Q_iQ_j}{4\pi\epsilon_0\epsilon_r r_{ij}}.$$

$Q_i,Q_j$ are charges in coulombs in this SI expression, $r_{ij}$ is in meters, $\epsilon_0$ is the vacuum permittivity, and $\epsilon_r$ is dimensionless; each pair energy is in joules. Molecular simulation packages usually express partial charges in units of the elementary charge and incorporate conversion factors into their formulas. Partial charges are model parameters and need not equal integer formal charges. Opposite signs attract; like signs repel. Force fields may modify this form or its pair scaling. See [OpenMM electrostatics](https://docs.openmm.org/latest/userguide/theory/02_standard_forces.html#coulomb-interaction-without-cutoff).

**Implementation check:** RDKit UFF has no explicit partial-charge Coulomb term; MMFF94 includes its own parameterized electrostatics. This distinction is visible in the [UFF force-field builder](https://github.com/rdkit/rdkit/blob/master/Code/GraphMol/ForceFieldHelpers/UFF/Builder.cpp) and [MMFF force-field builder](https://github.com/rdkit/rdkit/blob/master/Code/GraphMol/ForceFieldHelpers/MMFF/Builder.cpp). The generic decomposition above does not imply that every implementation includes every term.

#### 2.2.4.9. Additional Non-Bonded Terms (E<sub>additional</sub>)

Hydrogen bonding is often represented through electrostatics and van der Waals terms; adding a separate term without a matching parameterization can double-count it. Polarization, solvent, and directional hydrogen-bond terms require explicit model choices. A lower vacuum MM energy alone does not establish a solution-phase population or binding affinity.

### 2.2.5. Geometry Optimization

Geometry optimization moves coordinates downhill until convergence criteria are met. Starting from one structure typically finds a **nearby local minimum**, not necessarily the global minimum. Different starting conformers can reach different minima. A conformational search therefore generates multiple starting geometries, minimizes them consistently, and compares or clusters the results.

A small gradient indicates a stationary region; by itself it does not prove a minimum rather than a saddle point. Minimized potential energy also omits entropic and solvent contributions to thermodynamic stability. Local minimization is used to relax structures, not to generate a time-dependent trajectory. See [the OpenMM local-minimization discussion](https://docs.openmm.org/latest/userguide/theory/05_other_features.html#localenergyminimizer).

#### 2.2.5.1. Calculation of Energy Gradient

For Cartesian coordinate $q_i$, the gradient component and force are

$$g_i=(\nabla E)_i=\frac{\partial E}{\partial q_i},\qquad
F_i=-\frac{\partial E}{\partial q_i}=-g_i.$$

The gradient points uphill; the force points downhill. For coordinates in angstroms and energy in kcal/mol, both have units kcal/(mol $\cdot$ angstrom). RDKit's `CalcGrad()` returns **gradients**, not forces. A central finite difference provides an independent numerical check:

$$g_i\approx\frac{E(\mathbf q+h\mathbf e_i)-E(\mathbf q-h\mathbf e_i)}{2h}.$$

#### 2.2.5.2. Minimizing the Energy Function with Gradient Descent


A steepest-descent update is

$$\mathbf q^{(n+1)}=\mathbf q^{(n)}-\alpha\nabla E(\mathbf q^{(n)}).$$

The step size $\alpha$ must have compatible units and be small enough to decrease the energy; an overly large step can diverge. Practical optimizers can use line searches and curvature estimates instead of a fixed step size. RDKit's minimizer performs numerical optimization for us; the equation above explains the downhill direction, not its full algorithm.

Always examine the optimizer's **status**, the energy change, and the final geometry. `Minimize` returns 0 on convergence; a nonzero result must not be reported as a converged minimum. Increase the iteration limit or investigate the input rather than discarding that status.

### Research application: do three starting conformers give one answer?

Before using a geometry in a later quantum calculation, compare a few plausible conformers. Use **butane**, a four-carbon chain, so one central C–C–C–C torsion is easy to inspect. `anti` means the end groups are approximately opposite (180 degrees); the two `gauche` regions lie near +60 and −60 degrees.

Generate one reference graph, set three different starting torsions, and minimize each **independently with MMFF94**. We record the final torsion and terminal-carbon distance as well as energy. All values are calculated by the stated model. Three starts form a bounded demonstration, not an exhaustive conformational search.

**Predict:** can every minimization converge while the final geometries still differ? What information would an energy-only table hide?

In [ ]:
import pandas as pd

butane_reference = embed_smiles("CCCC")
butane_results, butane_conformers = [], []
for start_angle in (60.0, 180.0, 300.0):
    candidate = Chem.Mol(butane_reference)
    rdMolTransforms.SetDihedralDeg(candidate.GetConformer(), 0, 1, 2, 3, start_angle)
    candidate_ff = make_force_field(candidate, "MMFF94")
    initial_energy = candidate_ff.CalcEnergy()
    candidate_status = candidate_ff.Minimize(maxIts=500)
    if candidate_status != 0:
        raise RuntimeError(f"Butane start {start_angle} did not converge.")
    final_energy = candidate_ff.CalcEnergy()
    assert final_energy <= initial_energy + 1e-7
    final_torsion = rdMolTransforms.GetDihedralDeg(candidate.GetConformer(), 0, 1, 2, 3)
    terminal_distance = rdMolTransforms.GetBondLength(candidate.GetConformer(), 0, 3)
    butane_conformers.append(candidate)
    butane_results.append({"starting_torsion_deg": start_angle,
        "final_torsion_deg": final_torsion, "energy_kcal_mol": final_energy,
        "terminal_C_distance_A": terminal_distance, "status": candidate_status})
butane_table = pd.DataFrame(butane_results)
butane_table["relative_energy_kcal_mol"] = (butane_table.energy_kcal_mol
                                            - butane_table.energy_kcal_mol.min())
butane_table

In [ ]:
butane_labels = [f"start {value:g} deg" for value in butane_table.starting_torsion_deg]
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), constrained_layout=True)
axes[0].bar(butane_labels, butane_table.relative_energy_kcal_mol, color="#2563eb")
axes[0].set(ylabel="Energy above lowest sampled minimum (kcal/mol)",
            title="Same model; independent local minimizations")
axes[1].bar(butane_labels, butane_table.terminal_C_distance_A, color="#b45309")
axes[1].set(ylabel="Terminal carbon separation (angstrom)", ylim=(0, 4.6),
            title="Different shapes can satisfy convergence")
for index, row in butane_table.iterrows():
    axes[1].text(index, row.terminal_C_distance_A + 0.1,
                 f"torsion {row.final_torsion_deg:.0f} deg", ha="center", fontsize=9)
plt.show()
assert butane_table.status.eq(0).all()
assert np.ptp(butane_table.final_torsion_deg) > 100

**Decision and limit:** retain distinct low-energy conformers as candidates for subsequent calculations, with the model and starting geometries recorded. The lowest **sampled MMFF94** result is a defensible starting candidate; it is not proof of the global minimum, a solution population, or a binding pose. Similar energies for the two gauche results do not make their signed torsions identical.

**Guided exercise:** compare the final terminal distances for anti and gauche conformers. Explain why selecting only the most extended conformer is a geometric criterion, not a thermodynamic calculation. **Selected answer:** the anti result has more widely separated terminal carbons here. Turning relative MM energies into actual populations would require a defensible free-energy model and conformational sampling, beyond this minimization exercise.

#### Worked examples: optimizing cyclohexane and glucose

We retain a copy of each starting structure so the before/after comparison uses the same atoms and model.

First, deliberately perturb a generated cyclohexane structure and relax it with UFF:

In [ ]:
cyclohexane_start = embed_smiles("C1CCCCC1")
rng = np.random.default_rng(SEED)
conf = cyclohexane_start.GetConformer()
perturbed_xyz = conf.GetPositions() + rng.normal(scale=0.10, size=(cyclohexane_start.GetNumAtoms(), 3))
for atom_id, point in enumerate(perturbed_xyz):
    conf.SetAtomPosition(atom_id, point)
cyclohexane = Chem.Mol(cyclohexane_start)
uff = make_force_field(cyclohexane, "UFF")

In [ ]:
cyclohexane_initial_energy = uff.CalcEnergy()
print(f"Initial UFF energy: {cyclohexane_initial_energy:.4f} kcal/mol")

In [ ]:
uff_status = uff.Minimize(maxIts=1000)
cyclohexane_final_energy = uff.CalcEnergy()
print(f"UFF status: {uff_status} (0 = converged)")
print(f"Final UFF energy: {cyclohexane_final_energy:.4f} kcal/mol")
if uff_status != 0:
    raise RuntimeError("Cyclohexane did not converge within the iteration limit.")
assert cyclohexane_final_energy <= cyclohexane_initial_energy + 1e-8
fig = plt.figure(figsize=(9, 4))
plot_molecule(cyclohexane_start, fig.add_subplot(121, projection="3d"), "Cyclohexane: perturbed start")
plot_molecule(cyclohexane, fig.add_subplot(122, projection="3d"), "Cyclohexane: UFF local minimum")
plt.tight_layout()
plt.show()

Next, optimize **alpha-D-glucopyranose** with MMFF94. The inline isomeric SMILES below specifies the stereocenters and comes from [PubChem CID 79025](https://pubchem.ncbi.nlm.nih.gov/compound/Alpha-D-Glucose); it requires no online request when running the notebook.

A file name such as `glucose.pdb` does not validate identity or stereochemistry, and XYZ alone has no bond orders. For an existing, validated 3D structure, preserve its conformer when optimizing; calling `EmbedMolecule` would replace it with a newly generated geometry. Here embedding is intentional because our input is SMILES.

In [ ]:
alpha_glucose_smiles = "C([C@@H]1[C@H]([C@@H]([C@H]([C@H](O1)O)O)O)O)O"
glucose_start = embed_smiles(alpha_glucose_smiles)
assert rdMolDescriptors.CalcMolFormula(glucose_start) == "C6H12O6"
print("Reference isomeric SMILES:", Chem.MolToSmiles(Chem.RemoveHs(glucose_start)))
print("Assigned stereocenters:", Chem.FindMolChiralCenters(glucose_start, includeUnassigned=True))
glucose = Chem.Mol(glucose_start)
mmff = make_force_field(glucose, "MMFF94")

In [ ]:
glucose_initial_energy = mmff.CalcEnergy()
print(f"Initial MMFF94 energy: {glucose_initial_energy:.4f} kcal/mol")

In [ ]:
mmff_status = mmff.Minimize(maxIts=1000)
glucose_final_energy = mmff.CalcEnergy()
print(f"MMFF94 status: {mmff_status} (0 = converged)")
print(f"Final MMFF94 energy: {glucose_final_energy:.4f} kcal/mol")
if mmff_status != 0:
    raise RuntimeError("Glucose did not converge within the iteration limit.")
assert glucose_final_energy <= glucose_initial_energy + 1e-8
fig = plt.figure(figsize=(9, 4))
plot_molecule(glucose_start, fig.add_subplot(121, projection="3d"), "Glucose: embedded start")
plot_molecule(glucose, fig.add_subplot(122, projection="3d"), "Glucose: MMFF94 local minimum")
plt.tight_layout()
plt.show()

#### Recording a minimization trajectory

Use `MinimizeTrajectory` to record snapshots during one optimization. Repeatedly calling an optimizer for five iterations can restart its internal history, so it need not reproduce one continuous run. We start again from the saved initial glucose geometry and save approximately every five optimizer iterations.

These frames show **optimization progress, not molecular dynamics**: there is no physical time, temperature, or velocity. We plot frame number because the final snapshot can represent fewer than five additional iterations. See [RDKit trajectory objects](https://www.rdkit.org/docs/source/rdkit.Chem.rdtrajectory.html).

In [ ]:
trajectory_mol = Chem.Mol(glucose_start)
trajectory_ff = make_force_field(trajectory_mol, "MMFF94")
energy_values = [trajectory_ff.CalcEnergy()]
frames = [Chem.Mol(trajectory_mol)]
save_every = 5
trajectory_status, snapshots = trajectory_ff.MinimizeTrajectory(save_every, maxIts=1000)
if trajectory_status != 0:
    raise RuntimeError("The recorded minimization did not converge.")

# The Trajectory supplies dimensionality/atom-count context for its Snapshots.
trajectory = rdtrajectory.Trajectory(3, trajectory_mol.GetNumAtoms(), snapshots)
for frame_id in range(len(trajectory)):
    snapshot = trajectory.GetSnapshot(frame_id)
    frame = Chem.Mol(glucose_start)
    for atom_id in range(frame.GetNumAtoms()):
        frame.GetConformer().SetAtomPosition(atom_id, snapshot.GetPoint3D(atom_id))
    frames.append(frame)
    energy_values.append(snapshot.GetEnergy())

energy_values = np.asarray(energy_values)
assert len(frames) == len(energy_values) and len(frames) > 1
assert np.isfinite(energy_values).all()
assert np.all(np.diff(energy_values) <= 1e-6)
np.testing.assert_allclose(energy_values[-1], trajectory_ff.CalcEnergy(), atol=1e-6)
print(f"Saved {len(frames)} frames including the initial structure; status {trajectory_status}.")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(np.arange(len(energy_values)), energy_values - energy_values[-1], marker=".")
ax.set(xlabel="Saved frame (optimization progress, not time)",
       ylabel="Energy above final frame (kcal/mol)",
       title="Glucose MMFF94 minimization")
ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()

In [ ]:
selected_frames = [0, len(frames) // 2, len(frames) - 1]
fig = plt.figure(figsize=(12, 4))
for panel, frame_id in enumerate(selected_frames, start=1):
    plot_molecule(frames[frame_id], fig.add_subplot(1, 3, panel, projection="3d"),
                  f"Frame {frame_id}: {energy_values[frame_id]:.2f} kcal/mol")
plt.tight_layout()
plt.show()

#### Optional 3D animation and PDB export

The static frames above work offline. To animate them, combine PDB frames with `MODEL`/`ENDMDL` records and one final `END`. Use the explicit format `"pdb"` with `addModelsAsFrames`; `"trajectory"` is not a molecular file format. The animation interval controls playback only and is not a simulation timestep. The interactive display is disabled by default because it normally loads JavaScript from a CDN. See [3Dmol.js frame loading](https://3dmol.org/doc/GLViewer.html#addModelsAsFrames).

In [ ]:
def generate_pdb_trajectory(molecules):
    """Make one multi-model PDB string from frames with identical atom ordering."""
    if not molecules:
        raise ValueError("At least one frame is required.")
    records = []
    for model_id, mol in enumerate(molecules, start=1):
        records.append(f"MODEL     {model_id:4d}")
        # Remove each single-frame END record before adding model delimiters.
        for line in Chem.MolToPDBBlock(mol).splitlines():
            if line[:6].strip() in {"ATOM", "HETATM", "CONECT", "TER"}:
                records.append(line)
        records.append("ENDMDL")
    records.append("END")
    return "\n".join(records) + "\n"

In [ ]:
pdb_trajectory = generate_pdb_trajectory(frames)
assert sum(line.startswith("MODEL ") for line in pdb_trajectory.splitlines()) == len(frames)
assert sum(line == "END" for line in pdb_trajectory.splitlines()) == 1
print(f"Multi-model PDB prepared in memory: {len(frames)} models.")

In [ ]:
SHOW_INTERACTIVE = False
if SHOW_INTERACTIVE:
    import py3Dmol
    view = py3Dmol.view(width=700, height=400)
    view.addModelsAsFrames(pdb_trajectory, "pdb", {"keepH": True})
    view.setBackgroundColor("white")
    view.setStyle({"stick": {}, "sphere": {"scale": 0.25}})
    view.zoomTo()
    view.animate({"loop": "forward", "interval": 100})
    view.show()
else:
    print("Interactive animation disabled; the static frames show the optimization offline.")

### 2.2.6. Numerical self-check: gradient versus force

Stretch the C--C bond so the gradient is appreciable, then compare the analytic derivative of one Cartesian coordinate with an independent central finite difference. The input geometry stays unchanged by evaluating displaced coordinate arrays.

In [ ]:
gradient_probe = Chem.Mol(ethane)
rdMolTransforms.SetBondLength(gradient_probe.GetConformer(), carbon0, carbon1, 1.75)
probe_ff = make_force_field(gradient_probe, "UFF")
q = gradient_probe.GetConformer().GetPositions().ravel().copy()
gradient = np.asarray(probe_ff.CalcGrad())
coordinate = int(np.argmax(np.abs(gradient)))
h = 1e-5  # angstrom
q_plus, q_minus = q.copy(), q.copy()
q_plus[coordinate] += h
q_minus[coordinate] -= h
finite_difference = (probe_ff.CalcEnergy(q_plus.tolist()) - probe_ff.CalcEnergy(q_minus.tolist())) / (2*h)
np.testing.assert_allclose(gradient[coordinate], finite_difference, rtol=1e-5, atol=1e-5)
force = -gradient
print(f"Coordinate {coordinate}: derivative = {gradient[coordinate]:.6f}; finite difference = {finite_difference:.6f}")
print(f"Force component = {force[coordinate]:.6f} kcal/(mol angstrom)")
print("Energy, convergence, trajectory, scan, and gradient checks passed.")

### 2.2.7. Exercises and interpretation

1. Repeat the rigid ethane scan with MMFF94. Plot each model relative to **its own minimum**. Compare the curve shapes and energy ranges; do not compare the raw energy zeros of UFF and MMFF94.
2. Repeat cyclohexane embedding/minimization for seeds 1, 2, and 3. Are the final energies identical? Does finding the same energy three times prove a global minimum?
3. What happens if you minimize a distorted structure for only one iteration? Report both the status and energy. Why is a lower energy insufficient to claim convergence?
4. For the illustrative Lennard--Jones potential, calculate $E(\sigma)$ and $E(2^{1/6}\sigma)$. Explain what $\sigma$ and $\epsilon$ mean.
5. Explain why a trajectory of falling MM energies cannot be interpreted as a room-temperature molecular dynamics simulation or a reaction pathway.

**Answer hints:** (2) Repeated agreement provides evidence, not proof of the global minimum. (3) A nonzero status indicates that convergence was not achieved. (4) The pair energies are 0 and $-\epsilon$. (5) Minimization has no physical time integration or thermal sampling and keeps the molecular connectivity fixed.

### Optional audit of an older local structure file

Some working copies contain `structures/glucose.pdb`; it is not required by this notebook. If present, inspect its connectivity and compare its assigned stereochemistry with the explicitly chosen alpha-D-glucose reference. The formula alone cannot distinguish stereoisomers, and a mismatch is a reason to investigate the source rather than relabeling or silently using it. This audit preserves the file and its coordinates.

In [ ]:
legacy_path = Path("structures/glucose.pdb")
if legacy_path.is_file():
    # This legacy file has CONECT records: avoid adding extra bonds by proximity.
    legacy_mol = Chem.MolFromPDBFile(str(legacy_path), removeHs=False, proximityBonding=False)
    if legacy_mol is None:
        print("The local PDB could not be parsed; inspect it before use.")
    else:
        legacy_identity = Chem.MolToSmiles(Chem.RemoveHs(legacy_mol), isomericSmiles=True)
        reference_identity = Chem.MolToSmiles(Chem.MolFromSmiles(alpha_glucose_smiles), isomericSmiles=True)
        print("Local formula:", rdMolDescriptors.CalcMolFormula(legacy_mol))
        print("Local isomeric SMILES:", legacy_identity)
        print("Matches chosen alpha-D-glucose stereochemistry:", legacy_identity == reference_identity)
else:
    print("Optional legacy PDB is absent; the inline reference supplies the core example.")

### References and further reading

- [RDKit force-field helpers](https://www.rdkit.org/docs/source/rdkit.Chem.rdForceFieldHelpers.html): parameter coverage and minimization status values.
- [RDKit ForceField methods](https://www.rdkit.org/docs/source/rdkit.ForceField.rdForceField.html): energy units, gradients, and recorded minimization.
- [GROMACS bonded interactions](https://manual.gromacs.org/current/reference-manual/functions/bonded-interactions.html) and [OpenMM standard forces](https://docs.openmm.org/latest/userguide/theory/02_standard_forces.html): definitions and parameter conventions for illustrative potentials.
- [Original UFF paper](https://doi.org/10.1021/ja00051a040) and [original MMFF94 paper](https://onlinelibrary.wiley.com/doi/abs/10.1002/%28SICI%291096-987X%28199604%2917%3A5/6%3C490%3A%3AAID-JCC1%3E3.0.CO%3B2-P): scope and parameterization.
- [PubChem alpha-D-glucose, CID 79025](https://pubchem.ncbi.nlm.nih.gov/compound/Alpha-D-Glucose): provenance of the isomeric SMILES used here.

Next: [Chapter 3 -- Molecular Dynamics](Chapter03.ipynb).